# **1. Unzip Folder BULAI_JAGUNG dari Google Drive**

In [3]:
import zipfile

zip_path = "/content/drive/MyDrive/CornGuard/BULAI JAGUNG.v7i.yolov8.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("selesai, isi folder:", extract_path)

selesai, isi folder: /content/dataset


*Note: Dilakukan unzip dari Drive. Ini lebih stabil dibanding upload langsung ke Colab (hilang tiap sesi restart).*

# **2. Cek struktur dan data.yaml, pastikan urutan kelasnya masih sama seperti yang ditemukan sebelumnya.**

In [4]:
import os
import yaml

dataset_path = "/content/dataset"

# 1. Lihat struktur folder
print("=== Struktur folder ===")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:  # jangan tampilin isi file satu-satu, kepanjangan
        continue

# 2. Baca dan tampilkan data.yaml
print("\n=== Isi data.yaml ===")
yaml_path = os.path.join(dataset_path, "data.yaml")
with open(yaml_path) as f:
    data_yaml = yaml.safe_load(f)
print(data_yaml)

# 3. Hitung jumlah gambar dan label tiap split
print("\n=== Jumlah data per split ===")
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset_path, split, "images")
    lbl_dir = os.path.join(dataset_path, split, "labels")
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"{split}: images={n_img}, labels={n_lbl}")
    if n_img != n_lbl:
        print(f"  !! jumlah gambar dan label beda, perlu dicek")

# 4. Pastikan kelas Bulai memang index 0 (penting buat langkah filter berikutnya)
print("\n=== Cek urutan kelas ===")
print("names:", data_yaml.get("names"))
print("nc:", data_yaml.get("nc"))
if data_yaml.get("names", [])[0] != "PENYAKIT-BULAI-JAGUNG":
    print("!! PERINGATAN: urutan kelas beda dari yang diasumsikan, cek ulang sebelum filter label")
else:
    print("Aman, Bulai ada di index 0, sesuai asumsi sebelumnya")

=== Struktur folder ===
dataset/
  test/
    labels/
    images/
  train/
    labels/
    images/
  valid/
    labels/
    images/

=== Isi data.yaml ===
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['PENYAKIT-BULAI-JAGUNG', 'Sehat-Muda', 'sehat Tua'], 'roboflow': {'workspace': 'student-tqkot', 'project': 'bulai-jagung-oakym', 'version': 7, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/student-tqkot/bulai-jagung-oakym/dataset/7'}}

=== Jumlah data per split ===
train: images=3217, labels=3217
valid: images=322, labels=322
test: images=169, labels=169

=== Cek urutan kelas ===
names: ['PENYAKIT-BULAI-JAGUNG', 'Sehat-Muda', 'sehat Tua']
nc: 3
Aman, Bulai ada di index 0, sesuai asumsi sebelumnya


*NOTE: ['PENYAKIT-BULAI-JAGUNG', 'Sehat-Muda', 'sehat Tua'], dengan Bulai di index 0. Ini penting dicek ulang karena kalau urutannya beda, langkah filter di bawah bisa salah kelas yang kebuang.*

# **3. Filtering untuk variasi kelas yang ada didataset Bulai, sehingga hanya menyisakan NC = 1, dengan Label = 0 (Hanya Bulai, Sehat-Muda dan Sehat-Tua dibuang, persis dengan penjelasan pada Laporan Perancangan)**

In [6]:
import os

for split in ["train", "valid", "test"]:
    label_dir = f"/content/dataset/{split}/labels"
    class_ids_found = set()
    empty_count = 0

    for fname in os.listdir(label_dir):
        with open(os.path.join(label_dir, fname)) as f:
            content = f.read().strip()
        if content == "":
            empty_count += 1
        else:
            for line in content.splitlines():
                class_ids_found.add(line.split()[0])

    print(f"{split}: class-id yang tersisa = {class_ids_found}, file kosong = {empty_count}")

train: class-id yang tersisa = {'0'}, file kosong = 2027
valid: class-id yang tersisa = {'0'}, file kosong = 215
test: class-id yang tersisa = {'0'}, file kosong = 101


# **4. Memutuskan apakah foto yang kosong sebaiknya dibiarkan atau dibuang untuk mendapatkan dataset yang murni untuk gejala bulai saja.**

Buang foto (foto + label kosongnya dihapus dari dataset), paling simpel, dataset jadi murni cuma berisi foto yang ada gejala bulainya.

In [7]:
import os

removed_total = 0

for split in ["train", "valid", "test"]:
    img_dir = f"/content/dataset/{split}/images"
    label_dir = f"/content/dataset/{split}/labels"
    removed = 0

    for fname in os.listdir(label_dir):
        label_path = os.path.join(label_dir, fname)

        # cek apakah file label sekarang kosong (nggak ada baris Bulai sama sekali)
        with open(label_path) as f:
            content = f.read().strip()

        if content == "":
            # cari file gambar pasangannya (ekstensi bisa .jpg/.jpeg/.png, jadi dicek semua)
            base_name = os.path.splitext(fname)[0]
            for ext in [".jpg", ".jpeg", ".png"]:
                img_path = os.path.join(img_dir, base_name + ext)
                if os.path.exists(img_path):
                    os.remove(img_path)
                    break

            os.remove(label_path)
            removed += 1

    print(f"{split}: {removed} pasang gambar+label kosong dibuang")
    removed_total += removed

print(f"\nTotal dibuang: {removed_total}")

# cek ulang jumlah data setelah dibuang
print("\n=== Jumlah data setelah dibersihkan ===")
for split in ["train", "valid", "test"]:
    img_dir = f"/content/dataset/{split}/images"
    lbl_dir = f"/content/dataset/{split}/labels"
    print(f"{split}: images={len(os.listdir(img_dir))}, labels={len(os.listdir(lbl_dir))}")

train: 2027 pasang gambar+label kosong dibuang
valid: 215 pasang gambar+label kosong dibuang
test: 101 pasang gambar+label kosong dibuang

Total dibuang: 2343

=== Jumlah data setelah dibersihkan ===
train: images=1190, labels=1190
valid: images=107, labels=107
test: images=68, labels=68


Dari hasil output yang didapatkan:
1. Mayoritas foto di dataset BULAI_JAGUNG_v7i sebenarnya foto Sehat-Muda/Sehat-Tua, bukan foto bulai. Baru terlihat jelas sekarang setelah difilter bahwa jumlah 1190 (37%) adalah foto yang benar-benar jelas menunjukkan gejala bulai.
2. Rinciannya per split, proporsinya konsisten:
    *   train: 1190/3217 tersisa (37,0%)
    *   valid: 107/322 tersisa (33,2%)
    *   test: 68/169 tersisa (40,2%)
3. Hasil 1190 gambar (Gejala Bulai) akan dipakai karena masih tergolong layak untuk fine-tuning YOLO, meski lebih kecil dari 3217 (Masih ada foto Sehat-Muda dan Sehat-Tua).








# **5. Update data.yaml: ganti jadi nc: 1 dan names: ['Bulai'].**

*NC = Number of Classess, Variasi dari jumlah kelas yang ada di dataset.*

In [8]:
import yaml

yaml_path = "/content/dataset/data.yaml"

# baca dulu yang lama, biar path train/val/test-nya nggak perlu ditulis ulang manual
with open(yaml_path) as f:
    data_yaml = yaml.safe_load(f)

print("=== Sebelum diubah ===")
print(data_yaml)

# ubah cuma bagian kelas
data_yaml["nc"] = 1
data_yaml["names"] = ["Bulai"]

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("\n=== Sesudah diubah ===")
with open(yaml_path) as f:
    print(f.read())

=== Sebelum diubah ===
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['PENYAKIT-BULAI-JAGUNG', 'Sehat-Muda', 'sehat Tua'], 'roboflow': {'workspace': 'student-tqkot', 'project': 'bulai-jagung-oakym', 'version': 7, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/student-tqkot/bulai-jagung-oakym/dataset/7'}}

=== Sesudah diubah ===
names:
- Bulai
nc: 1
roboflow:
  license: CC BY 4.0
  project: bulai-jagung-oakym
  url: https://universe.roboflow.com/student-tqkot/bulai-jagung-oakym/dataset/7
  version: 7
  workspace: student-tqkot
test: ../test/images
train: ../train/images
val: ../valid/images

